# LLM Classification Finetuning - Baseline (Kaggle)
This notebook is self-contained and writes `submission.csv` to `/kaggle/working/`.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/)
# that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [ ]:
import ast
import re
from dataclasses import dataclass
from typing import List, Tuple, Optional
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split


In [ ]:
@dataclass
class DataPaths:
    train_path: str
    test_path: str


class DataLoader:
    def __init__(self, paths: DataPaths):
        self.paths = paths

    def load(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        train = pd.read_csv(self.paths.train_path)
        test = pd.read_csv(self.paths.test_path)
        return train, test


class TextParser:
    @staticmethod
    def parse_list(text):
        if not isinstance(text, str):
            return []
        try:
            return ast.literal_eval(text)
        except Exception:
            return []

    @staticmethod
    def normalize(text: str) -> str:
        if not isinstance(text, str):
            return ""
        text = text.strip()
        text = re.sub(r"\s+", " ", text)
        return text

    def join_turns(self, text):
        turns = self.parse_list(text)
        if not turns:
            return ""
        joined = " <TURN> ".join(turns)
        return self.normalize(joined)


class FeatureBuilder:
    def __init__(self):
        self.parser = TextParser()

    def add_text_fields(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["prompt_text"] = df["prompt"].apply(self.parser.join_turns)
        df["resp_a_text"] = df["response_a"].apply(self.parser.join_turns)
        df["resp_b_text"] = df["response_b"].apply(self.parser.join_turns)
        df["pair_text"] = "A: " + df["resp_a_text"] + " <SEP> B: " + df["resp_b_text"]
        return df

    def add_numeric_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        def count_tokens(s):
            if not s:
                return 0
            return len(s.split())

        for col in ["prompt_text", "resp_a_text", "resp_b_text"]:
            df[col + "_n_chars"] = df[col].str.len()
            df[col + "_n_tokens"] = df[col].apply(count_tokens)

        df["len_diff_chars"] = df["resp_a_text_n_chars"] - df["resp_b_text_n_chars"]
        df["len_diff_tokens"] = df["resp_a_text_n_tokens"] - df["resp_b_text_n_tokens"]
        df["abs_len_diff_chars"] = df["len_diff_chars"].abs()
        df["abs_len_diff_tokens"] = df["len_diff_tokens"].abs()
        return df


class TextVectorizer:
    def __init__(self, max_features=50000, ngram_range=(1, 2), min_df=3):
        self.vectorizer = {
            "prompt_text": TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=min_df),
            "resp_a_text": TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=min_df),
            "resp_b_text": TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=min_df),
            "pair_text": TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=min_df),
        }

    def fit_transform(self, df: pd.DataFrame):
        mats = []
        for col, vec in self.vectorizer.items():
            mats.append(vec.fit_transform(df[col]))
        return hstack(mats)

    def transform(self, df: pd.DataFrame):
        mats = []
        for col, vec in self.vectorizer.items():
            mats.append(vec.transform(df[col]))
        return hstack(mats)


class DatasetBuilder:
    def __init__(self, feature_cols: List[str]):
        self.feature_cols = feature_cols
        self.vectorizer = TextVectorizer()

    def build_train(self, df: pd.DataFrame):
        X_text = self.vectorizer.fit_transform(df)
        X_num = csr_matrix(df[self.feature_cols].values)
        X = hstack([X_text, X_num])
        y = df[["winner_model_a", "winner_model_b", "winner_tie"]].values
        return X, y

    def build_test(self, df: pd.DataFrame):
        X_text = self.vectorizer.transform(df)
        X_num = csr_matrix(df[self.feature_cols].values)
        X = hstack([X_text, X_num])
        return X


@dataclass
class LogRegConfig:
    max_iter: int = 200
    C: float = 1.0
    n_jobs: int = -1
    random_state: int = 42
    test_size: float = 0.2
    verbose: int = 1


class LogRegModel:
    def __init__(self, config: Optional[LogRegConfig] = None):
        self.config = config or LogRegConfig()
        self.model = LogisticRegression(
            max_iter=self.config.max_iter,
            C=self.config.C,
            n_jobs=self.config.n_jobs,
            multi_class="multinomial",
            solver="saga",
            random_state=self.config.random_state,
            verbose=self.config.verbose,
        )

    def _to_labels(self, y: np.ndarray) -> np.ndarray:
        return np.argmax(y, axis=1)

    def fit(self, X, y: np.ndarray):
        y_labels = self._to_labels(y)
        self.model.fit(X, y_labels)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def train_eval_predict(self, X, y: np.ndarray, X_test):
        y_labels = self._to_labels(y)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X,
            y,
            test_size=self.config.test_size,
            random_state=self.config.random_state,
            stratify=y_labels,
        )
        self.fit(X_tr, y_tr)
        val_proba = self.predict_proba(X_val)
        loss = log_loss(y_val, val_proba)

        self.fit(X, y)
        test_proba = self.predict_proba(X_test)
        return loss, test_proba


class SubmissionBuilder:
    def __init__(self, id_col: str = "id"):
        self.id_col = id_col
        self.target_cols = ["winner_model_a", "winner_model_b", "winner_tie"]

    def build(self, test_df: pd.DataFrame, proba, path: str = "submission.csv"):
        if len(proba) != len(test_df):
            raise ValueError("proba rows must match test_df rows")

        sub = pd.DataFrame(proba, columns=self.target_cols)
        sub.insert(0, self.id_col, test_df[self.id_col].values)
        sub.to_csv(path, index=False)
        return sub


In [ ]:
def find_dataset_dir(root='/kaggle/input'):
    for dirname, _, filenames in os.walk(root):
        if 'train.csv' in filenames and 'test.csv' in filenames:
            return dirname
    raise FileNotFoundError('Could not find train.csv/test.csv in /kaggle/input')

DATA_DIR = find_dataset_dir()
print('Using data dir:', DATA_DIR)

In [ ]:
paths = DataPaths(
    train_path=os.path.join(DATA_DIR, 'train.csv'),
    test_path=os.path.join(DATA_DIR, 'test.csv')
)

loader = DataLoader(paths)
train_df, test_df = loader.load()

builder = FeatureBuilder()
train_df = builder.add_text_fields(train_df)
train_df = builder.add_numeric_features(train_df)

test_df = builder.add_text_fields(test_df)
test_df = builder.add_numeric_features(test_df)

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)

In [ ]:
feature_cols = [
    'prompt_text_n_chars', 'resp_a_text_n_chars', 'resp_b_text_n_chars',
    'prompt_text_n_tokens', 'resp_a_text_n_tokens', 'resp_b_text_n_tokens',
    'len_diff_chars', 'len_diff_tokens',
    'abs_len_diff_chars', 'abs_len_diff_tokens',
]

dataset = DatasetBuilder(feature_cols=feature_cols)
X_train, y_train = dataset.build_train(train_df)
X_test = dataset.build_test(test_df)

In [ ]:
model = LogRegModel(LogRegConfig(verbose=1))
val_loss, test_proba = model.train_eval_predict(X_train, y_train, X_test)
print('val log loss:', val_loss)

In [ ]:
sub_builder = SubmissionBuilder()
sub_builder.build(test_df, test_proba, path='submission.csv')
print('submission.csv saved to /kaggle/working/')